# Train WikiANN NER Final Model

This notebook trains a final Named Entity Recognition model for the backend API using the HuggingFace WikiANN dataset.

The target entities are:
- `PERSON`
- `ORGANIZATION`
- `LOCATION`

The default language is English (`LANGUAGE = "en"`). The model is `xlm-roberta-base`, trained with PyTorch and HuggingFace `Trainer`.

This notebook is configured for full WikiANN training on Windows. It saves the final model under `backends/app/models/saved_ner_model/`.

## 1. Install Libraries

Run this cell once in the active Jupyter environment if the required packages are not installed.

In [ ]:
import subprocess
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
requirements_candidates = [
    cwd / "requirements.txt",
    cwd.parent / "requirements.txt",
    cwd / "backends" / "requirements.txt",
]
requirements_path = next((path for path in requirements_candidates if path.exists()), None)
if requirements_path is None:
    raise FileNotFoundError("Could not find backends/requirements.txt from the current notebook directory.")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])


## 2. Import Libraries

In [ ]:
import inspect
import json
import os
from pathlib import Path

import evaluate
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    pipeline,
)

## 3. Config

In [ ]:
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

LANGUAGE = "en"
MODEL_CHECKPOINT = "xlm-roberta-base"

USE_SUBSET = False
SUBSET_TRAIN_SIZE = 500
SUBSET_VALIDATION_SIZE = 100
SUBSET_TEST_SIZE = 100
MAX_LENGTH = 128

EPOCHS = 3
BATCH_SIZE = 8 if torch.cuda.is_available() else 2
GRADIENT_ACCUMULATION_STEPS = 1
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
FP16 = torch.cuda.is_available()

cwd = Path.cwd().resolve()
if cwd.name == "notebooks" and cwd.parent.name == "backends":
    PROJECT_ROOT = cwd.parent.parent
elif cwd.name == "backends":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

BACKEND_DIR = PROJECT_ROOT / "backends"
REPORTS_DIR = BACKEND_DIR / "reports"
MODEL_SAVE_DIR = BACKEND_DIR / "app" / "models" / "saved_ner_model"
TRAINING_OUTPUT_DIR = BACKEND_DIR / "training_outputs" / "wikiann_ner"
METRICS_PATH = REPORTS_DIR / "metrics.json"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Language: {LANGUAGE}")
print(f"Model checkpoint: {MODEL_CHECKPOINT}")
print(f"Using subset: {USE_SUBSET}")
print(f"Epochs: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Metrics path: {METRICS_PATH}")
print(f"Model save path: {MODEL_SAVE_DIR}")

## 4. Load Dataset

WikiANN is loaded from HuggingFace Datasets. The default language is English.

In [ ]:
dataset = load_dataset("wikiann", LANGUAGE)

if USE_SUBSET:
    dataset["train"] = dataset["train"].select(range(min(SUBSET_TRAIN_SIZE, len(dataset["train"]))))
    dataset["validation"] = dataset["validation"].select(range(min(SUBSET_VALIDATION_SIZE, len(dataset["validation"]))))
    dataset["test"] = dataset["test"].select(range(min(SUBSET_TEST_SIZE, len(dataset["test"]))))

dataset

## 5. Inspect Sample and Labels

In [ ]:
label_names = dataset["train"].features["ner_tags"].feature.names
num_labels = len(label_names)

print("Labels:")
for idx, label in enumerate(label_names):
    print(f"{idx}: {label}")

sample = dataset["train"][0]
print("\nSample tokens:")
print(sample["tokens"])
print("\nSample tag ids:")
print(sample["ner_tags"])
print("\nSample tag names:")
print([label_names[tag_id] for tag_id in sample["ner_tags"]])

## 6. Preprocessing

The dataset already provides tokenized text in the `tokens` field and token-level NER labels in `ner_tags`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

print(f"Tokenizer: {tokenizer.__class__.__name__}")
print(f"Fast tokenizer: {tokenizer.is_fast}")

## 7. Tokenization

`is_split_into_words=True` is required because WikiANN examples are already split into word tokens.

In [ ]:
tokenized_sample = tokenizer(sample["tokens"], is_split_into_words=True)

print(tokenizer.convert_ids_to_tokens(tokenized_sample["input_ids"]))
print(tokenized_sample.word_ids())

## 8. Label Alignment

NER labels must be aligned from word-level labels to subword-level tokenizer outputs.

Rules used here:
- Label only the first subword of each original word.
- Assign `-100` to later subwords so the loss ignores them.
- Assign `-100` to special tokens.

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True,
    )

    aligned_labels = []
    for example_index, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=example_index)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

In [ ]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

tokenized_dataset

## 9. Build `id2label` and `label2id`

In [ ]:
id2label = {idx: label for idx, label in enumerate(label_names)}
label2id = {label: idx for idx, label in id2label.items()}

print("id2label:", id2label)
print("label2id:", label2id)

## 10. Create Model

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

model.gradient_checkpointing_enable()
model.config.use_cache = False

mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
device = "cuda" if torch.cuda.is_available() else "mps" if mps_available else "cpu"
print(f"Available device: {device}")

## 11. Data Collator

In [ ]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

## 12. Metrics

`seqeval` computes sequence labeling metrics. Positions with label `-100` are ignored.

In [ ]:
seqeval = evaluate.load("seqeval")

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []

    for prediction_row, label_row in zip(predictions, labels):
        row_predictions = []
        row_labels = []

        for prediction_id, label_id in zip(prediction_row, label_row):
            if label_id == -100:
                continue
            row_predictions.append(id2label[int(prediction_id)])
            row_labels.append(id2label[int(label_id)])

        true_predictions.append(row_predictions)
        true_labels.append(row_labels)

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

## 13. Training

In [ ]:
training_args_kwargs = {
    "output_dir": str(TRAINING_OUTPUT_DIR),
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": BATCH_SIZE,
    "per_device_eval_batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "num_train_epochs": EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "evaluation_strategy": "epoch",
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1",
    "greater_is_better": True,
    "fp16": FP16,
    "dataloader_num_workers": 0,
    "overwrite_output_dir": True,
    "save_total_limit": 2,
    "logging_strategy": "steps",
    "logging_steps": 100,
    "report_to": "none",
}

training_args_signature = inspect.signature(TrainingArguments.__init__).parameters
if "evaluation_strategy" not in training_args_signature and "eval_strategy" in training_args_signature:
    training_args_kwargs["eval_strategy"] = training_args_kwargs.pop("evaluation_strategy")

training_args_kwargs = {
    key: value
    for key, value in training_args_kwargs.items()
    if key in training_args_signature
}

training_args = TrainingArguments(**training_args_kwargs)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_dataset["train"],
    "eval_dataset": tokenized_dataset["validation"],
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

trainer_signature = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_signature:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_signature:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

trainer.train()

## 14. Evaluation

In [ ]:
validation_metrics = trainer.evaluate(tokenized_dataset["validation"])
test_metrics = trainer.evaluate(tokenized_dataset["test"], metric_key_prefix="test")

print("Validation metrics:")
print(validation_metrics)
print("\nTest metrics:")
print(test_metrics)

## 15. Save Metrics

Metrics are saved to `backends/reports/metrics.json`.

In [ ]:
metrics_payload = {
    "language": LANGUAGE,
    "model_checkpoint": MODEL_CHECKPOINT,
    "use_subset": USE_SUBSET,
    "subset_train_size": SUBSET_TRAIN_SIZE if USE_SUBSET else None,
    "subset_validation_size": SUBSET_VALIDATION_SIZE if USE_SUBSET else None,
    "subset_test_size": SUBSET_TEST_SIZE if USE_SUBSET else None,
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "labels": label_names,
    "validation": validation_metrics,
    "test": test_metrics,
}

with METRICS_PATH.open("w", encoding="utf-8") as metrics_file:
    json.dump(metrics_payload, metrics_file, indent=2)

print(f"Saved metrics to: {METRICS_PATH}")

## 16. Save Model

The trained model and tokenizer are saved to `backends/app/models/saved_ner_model/` for later backend API loading.

In [ ]:
trainer.save_model(str(MODEL_SAVE_DIR))
tokenizer.save_pretrained(str(MODEL_SAVE_DIR))

print(f"Saved model and tokenizer to: {MODEL_SAVE_DIR}")

## 17. Final Inference Test

In [ ]:
ner_pipeline = pipeline(
    task="token-classification",
    model=str(MODEL_SAVE_DIR),
    tokenizer=str(MODEL_SAVE_DIR),
    aggregation_strategy="simple",
)

test_text = "Barack Obama visited Microsoft headquarters in Seattle."
predictions = ner_pipeline(test_text)

print(test_text)
predictions